# Python程序设计（项目实战） · 工单P4.1《多源数据采集系统》 — 教师参考版
第6周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 最小三线程采集演示（完整版见 starter p41_collector.py）

In [ ]:

# --- P4.1 最小三线程演示: 采集→Queue→处理→统计(运行约10秒) ---
import subprocess, sys, time, atexit, queue, threading
ps = [subprocess.Popen([sys.executable,
        "../02-程序设计基础/虚拟产线/mini_modbus_server.py", str(p), str(u)])
      for p, u in [(5020, 1), (5021, 2), (5022, 3)]]
atexit.register(lambda: [p.terminate() for p in ps])
time.sleep(1.2)
from pymodbus.client import ModbusTcpClient

q = queue.Queue(maxsize=1000)
stat = {"ok": 0, "fail": 0, "got": 0}
stop = threading.Event()

def collect():
    clients = [ModbusTcpClient(host="127.0.0.1", port=p) for p in (5020, 5021, 5022)]
    [c.connect() for c in clients]
    while not stop.is_set():
        for c in clients:
            try:
                rr = c.read_holding_registers(address=0, count=5)
                q.put(rr.registers, timeout=1); stat["ok"] += 1
            except Exception:
                stat["fail"] += 1
        time.sleep(0.5)

def process():
    while not stop.is_set() or not q.empty():
        try:
            rows = q.get(timeout=1); stat["got"] += 1
        except queue.Empty:
            pass

t1, t2 = threading.Thread(target=collect), threading.Thread(target=process)
t1.start(); t2.start(); time.sleep(8); stop.set(); t1.join(); t2.join()
total = stat["ok"] + stat["fail"]
print(f"采集率: {stat['ok']}/{total} = {stat['ok']/max(total,1):.1%} | 入队处理 {stat['got']} 条")
[p.terminate() for p in ps]
print("演示完成; 完整三线程系统(断线重连/入库)见 学生代码框架/p41_collector.py")
